# Лабораторная работа: PostgreSQL / БД «Авиаперевозки»

Этот ноутбук подготовлен для **локального запуска через Docker**.

Что уже настроено:
- PostgreSQL поднимается локально в контейнере;
- demo-база «Авиаперевозки» загружается автоматически при первой инициализации;
- ноутбук подключается к `localhost`, а не к внешнему серверу;
- SQL-решения по заданиям уже вставлены в ноутбук.

> Если Docker уже занят другим PostgreSQL на порту 5432, в проекте используется порт `5433` на хост-машине.


> Важно: запускай ноутбук **сверху вниз** через **Run All**. В начале уже добавлены все переменные окружения и вспомогательные функции `get_conn()`, `query_df()` и `execute_scalar()`.


In [1]:
import os
import json
import psycopg2
import pandas as pd
from psycopg2.extras import DictCursor
from IPython.display import display, Markdown

try:
    from dotenv import load_dotenv
    load_dotenv()
except Exception:
    pass

POSTGRESQL_HOST = os.getenv('POSTGRESQL_HOST', 'localhost')
POSTGRESQL_PORT = int(os.getenv('POSTGRESQL_PORT', '5433'))
POSTGRESQL_DB = os.getenv('POSTGRESQL_DB', 'demo')
POSTGRESQL_USER = os.getenv('POSTGRESQL_USER', 'demo')
POSTGRESQL_PASSWORD = os.getenv('POSTGRESQL_PASSWORD', 'demo')


def get_conn():
    conn = psycopg2.connect(
        dbname=POSTGRESQL_DB,
        user=POSTGRESQL_USER,
        password=POSTGRESQL_PASSWORD,
        host=POSTGRESQL_HOST,
        port=POSTGRESQL_PORT,
    )
    with conn.cursor() as cur:
        cur.execute('SET search_path TO bookings, public;')
    return conn


def query_df(sql: str):
    with get_conn() as conn:
        with conn.cursor(cursor_factory=DictCursor) as cur:
            cur.execute('SET search_path TO bookings, public;')
            cur.execute(sql)
            if cur.description is None:
                return pd.DataFrame()
            rows = cur.fetchall()
            if not rows:
                return pd.DataFrame(columns=[desc[0] for desc in cur.description])
            return pd.DataFrame([dict(r) for r in rows])


def execute_scalar(sql: str):
    with get_conn() as conn:
        with conn.cursor() as cur:
            cur.execute('SET search_path TO bookings, public;')
            cur.execute(sql)
            row = cur.fetchone()
            return row[0] if row else None


def print_sql(title: str, sql: str):
    display(Markdown(f'### {title}'))
    print(sql.strip())


In [2]:
connection_info = {
    'POSTGRESQL_HOST': POSTGRESQL_HOST,
    'POSTGRESQL_PORT': POSTGRESQL_PORT,
    'POSTGRESQL_DB': POSTGRESQL_DB,
    'POSTGRESQL_USER': POSTGRESQL_USER,
}
connection_info

{'POSTGRESQL_HOST': 'localhost',
 'POSTGRESQL_PORT': 5433,
 'POSTGRESQL_DB': 'demo',
 'POSTGRESQL_USER': 'demo'}

In [3]:
print('Подключение будет выполнено к:', f"{POSTGRESQL_HOST}:{POSTGRESQL_PORT}/{POSTGRESQL_DB}")

Подключение будет выполнено к: localhost:5433/demo


## 2. Проверка подключения

In [4]:
query_df('SELECT * FROM seats LIMIT 5')

,aircraft_code,seat_no,fare_conditions
0,319,2A,Business
1,319,2C,Business
2,319,2D,Business
3,319,2F,Business
4,319,3A,Business


## 3. Список таблиц

In [5]:
with get_conn() as conn:
    conn.get_dsn_parameters()

In [6]:
# Получаем список таблиц схемы bookings
tables_df = query_df("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bookings'
    ORDER BY table_name;
""")
tables_df


,table_name
0,aircrafts
1,aircrafts_data
2,airports
3,airports_data
4,boarding_passes
5,bookings
6,flights
7,flights_v
8,routes
9,seats


## Задание 1. Структура каждой таблицы и краткое описание

Ниже приведено краткое описание основных таблиц демонстрационной БД.

- **aircrafts_data** — справочник моделей самолётов: код самолёта, модель, максимальная дальность полёта.
- **airports_data** — справочник аэропортов: код аэропорта, название, город, координаты, часовой пояс.
- **boarding_passes** — посадочные талоны: билет, рейс, номер посадки, место.
- **bookings** — бронирования: номер бронирования, дата бронирования, общая сумма.
- **flights** — рейсы: идентификатор рейса, номер, расписание, фактические времена, аэропорты, статус рейса, самолёт.
- **seats** — места в самолётах: код самолёта, место, класс обслуживания.
- **ticket_flights** — сегменты перелёта по билету: билет, рейс, тариф, стоимость.
- **tickets** — билеты пассажиров: номер билета, бронирование, пассажир, контакты.

In [7]:
sql_air_data = """
SELECT * FROM aircrafts_data;

"""
print_sql('SQL для задания', sql_air_data)
query_df(sql_air_data)


### SQL для задания

SELECT * FROM aircrafts_data;


,aircraft_code,model,range
0,773,"{'en': 'Boeing 777-300', 'ru': '?????????? 777...",11100
1,763,"{'en': 'Boeing 767-300', 'ru': '?????????? 767...",7900
2,SU9,"{'en': 'Sukhoi Superjet-100', 'ru': '?????????...",3000
3,320,"{'en': 'Airbus A320-200', 'ru': '?????????????...",5700
4,321,"{'en': 'Airbus A321-200', 'ru': '?????????????...",5600
5,319,"{'en': 'Airbus A319-100', 'ru': '?????????????...",6700
6,733,"{'en': 'Boeing 737-300', 'ru': '?????????? 737...",4200
7,CN1,"{'en': 'Cessna 208 Caravan', 'ru': '??????????...",1200
8,CR2,"{'en': 'Bombardier CRJ-200', 'ru': '??????????...",2700


In [8]:
sql_aircafts = """
SELECT * FROM aircrafts;
"""
query_df(sql_aircafts)

,aircraft_code,model,range
0,773,Boeing 777-300,11100
1,763,Boeing 767-300,7900
2,SU9,Sukhoi Superjet-100,3000
3,320,Airbus A320-200,5700
4,321,Airbus A321-200,5600
5,319,Airbus A319-100,6700
6,733,Boeing 737-300,4200
7,CN1,Cessna 208 Caravan,1200
8,CR2,Bombardier CRJ-200,2700


In [9]:
for table_name in tables_df['table_name']:
    print(f'\n=== {table_name} ===')
    display(query_df(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'bookings' AND table_name = '{table_name}'
        ORDER BY ordinal_position;
    """))


=== aircrafts ===


,column_name,data_type
0,aircraft_code,character
1,model,text
2,range,integer



=== aircrafts_data ===


,column_name,data_type
0,aircraft_code,character
1,model,jsonb
2,range,integer



=== airports ===


...

,column_name,data_type
0,aircraft_code,character
1,seat_no,character varying
2,fare_conditions,character varying



=== ticket_flights ===


,column_name,data_type
0,ticket_no,character
1,flight_id,integer
2,fare_conditions,character varying
3,amount,numeric



=== tickets ===


,column_name,data_type
0,ticket_no,character
1,book_ref,character
2,passenger_id,character varying
3,passenger_name,text
4,contact_data,jsonb


колво всех аэропортов

In [10]:
sql_o = """ 
SELECT * FROM airports;
"""
query_df(sql_o)
print(sql_o)

 
SELECT * FROM airports;



In [11]:
sql_d = """ 
SELECT COUNT(DISTINCT airport_code)
FROM airports_data;
"""
query_df(sql_d)


,count
0,104


вывести самолет, у которого самое маленькое колво мест

In [12]:
sql_x = """ 
SELECT 
    a.aircraft_code,
    a.model->>'en' AS model_name,
    COUNT(*) AS seats_count
FROM seats s
JOIN aircrafts_data a ON a.aircraft_code = s.aircraft_code
GROUP BY a.aircraft_code, a.model
ORDER BY seats_count ASC
LIMIT 1;;
"""
query_df(sql_x)

,aircraft_code,model_name,seats_count
0,CN1,Cessna 208 Caravan,12


In [15]:
sql_un = """
SELECT DISTINCT status FROM flights;
"""
query_df(sql_un)


,status
0,Departed
1,Arrived
2,On Time
3,Cancelled
4,Delayed
5,Scheduled


## Задание 2. Типы столбцов и количество записей

In [9]:
table_stats = {}
for table_name in tables_df['table_name']:
    table_stats[table_name] = execute_scalar(f'SELECT COUNT(*) FROM {table_name}')

stats_df = pd.DataFrame([{'table_name': k, 'row_count': v} for k, v in table_stats.items()])\
    .sort_values(['row_count', 'table_name'], ascending=[False, True])
stats_df

,table_name,row_count
10,ticket_flights,1045726
4,boarding_passes,579686
11,tickets,366733
5,bookings,262788
6,flights,33121
7,flights_v,33121
9,seats,1339
8,routes,710
2,airports,104
3,airports_data,104


In [10]:
table_stats

{'aircrafts': 9,
 'aircrafts_data': 9,
 'airports': 104,
 'airports_data': 104,
 'boarding_passes': 579686,
 'bookings': 262788,
 'flights': 33121,
 'flights_v': 33121,
 'routes': 710,
 'seats': 1339,
 'ticket_flights': 1045726,
 'tickets': 366733}

In [11]:
max_table = stats_df.iloc[0]
print(f"Таблица с максимальным числом записей: {max_table['table_name']}")
print(f"Количество записей: {max_table['row_count']}")

Таблица с максимальным числом записей: ticket_flights
Количество записей: 1045726


In [33]:
for table_name in tables_df['table_name']:
    print(f'\n=== {table_name}: типы столбцов ===')
    display(query_df(f"""
        SELECT column_name, udt_name AS pg_type
        FROM information_schema.columns
        WHERE table_schema = 'bookings' AND table_name = '{table_name}'
        ORDER BY ordinal_position;
    """))


=== aircrafts: типы столбцов ===


,column_name,pg_type
0,aircraft_code,bpchar
1,model,text
2,range,int4



=== aircrafts_data: типы столбцов ===


,column_name,pg_type
0,aircraft_code,bpchar
1,model,jsonb
2,range,int4



=== airports: типы столбцов ===


...

,column_name,pg_type
0,aircraft_code,bpchar
1,seat_no,varchar
2,fare_conditions,varchar



=== ticket_flights: типы столбцов ===


,column_name,pg_type
0,ticket_no,bpchar
1,flight_id,int4
2,fare_conditions,varchar
3,amount,numeric



=== tickets: типы столбцов ===


,column_name,pg_type
0,ticket_no,bpchar
1,book_ref,bpchar
2,passenger_id,varchar
3,passenger_name,text
4,contact_data,jsonb


In [35]:
sql_o = """ 
SELECT * FROM tickets;
"""
query_df(sql_o)
print(sql_o)

 
SELECT * FROM tickets;



## Задание 3. Названия тарифов, которые предлагают авиаперевозчики пассажирам

In [13]:
sql_task_3 = """
SELECT DISTINCT fare_conditions
FROM ticket_flights
ORDER BY fare_conditions;
"""
print_sql('SQL для задания 3', sql_task_3)
query_df(sql_task_3)

### SQL для задания 3

SELECT DISTINCT fare_conditions
FROM ticket_flights
ORDER BY fare_conditions;


,fare_conditions
0,Business
1,Comfort
2,Economy


## Задание 4. Общая сумма выручки по каждому тарифу

In [14]:
sql_task_4 = """
SELECT
    fare_conditions,
    SUM(amount) AS total_revenue
FROM ticket_flights
GROUP BY fare_conditions
ORDER BY total_revenue DESC;
"""
print_sql('SQL для задания 4', sql_task_4)
query_df(sql_task_4)

### SQL для задания 4

SELECT
    fare_conditions,
    SUM(amount) AS total_revenue
FROM ticket_flights
GROUP BY fare_conditions
ORDER BY total_revenue DESC;


,fare_conditions,total_revenue
0,Economy,14695684400.00
1,Business,5505179600.00
2,Comfort,566116900.00


## Задание 5. Какой тариф приносит максимальный доход?

In [20]:
sql_task_5 = """
SELECT
    fare_conditions,
    SUM(amount) AS total_revenue
FROM ticket_flights
GROUP BY fare_conditions
ORDER BY total_revenue DESC
LIMIT 1;
"""
print_sql('SQL для задания 5', sql_task_5)
query_df(sql_task_5)

### SQL для задания 5

SELECT
    fare_conditions,
    SUM(amount) AS total_revenue
FROM ticket_flights
GROUP BY fare_conditions
ORDER BY total_revenue DESC
LIMIT 1;


,fare_conditions,total_revenue
0,Economy,14695684400.00


## Блок про время выполнения запросов

В исходном ноутбуке использовалась таблица `departments`, но в БД «Авиаперевозки» её нет. Поэтому ниже показан рабочий пример измерения времени на существующей таблице `flights`.

In [16]:
%%time
sample_df = query_df('SELECT * FROM flights LIMIT 1000')
sample_df.head()

CPU times: total: 78.1 ms
Wall time: 71.9 ms


,flight_id,flight_no,scheduled_departure,scheduled_arrival,departure_airport,arrival_airport,status,aircraft_code,actual_departure,actual_arrival
0,1185,PG0134,2017-09-10 06:50:00+00:00,2017-09-10 11:55:00+00:00,DME,BTK,Scheduled,319,NaT,NaT
1,3979,PG0052,2017-08-25 11:50:00+00:00,2017-08-25 14:35:00+00:00,VKO,HMA,Scheduled,CR2,NaT,NaT
2,4739,PG0561,2017-09-05 09:30:00+00:00,2017-09-05 11:15:00+00:00,VKO,AER,Scheduled,763,NaT,NaT
3,5502,PG0529,2017-09-12 06:50:00+00:00,2017-09-12 08:20:00+00:00,SVO,UFA,Scheduled,763,NaT,NaT
4,6938,PG0461,2017-09-04 09:25:00+00:00,2017-09-04 10:20:00+00:00,SVO,ULV,Scheduled,SU9,NaT,NaT


## Задание 6. Найти модель самолёта с минимальной максимальной дальностью полёта двумя способами

In [17]:
sql_task_6_a = """
SELECT model, range
FROM aircrafts_data
WHERE range = (SELECT MIN(range) FROM aircrafts_data);
"""

sql_task_6_b = """
SELECT model, range
FROM aircrafts_data
ORDER BY range ASC
LIMIT 1;
"""

print_sql('Способ 1', sql_task_6_a)
print_sql('Способ 2', sql_task_6_b)

### Способ 1

SELECT model, range
FROM aircrafts_data
WHERE range = (SELECT MIN(range) FROM aircrafts_data);


### Способ 2

SELECT model, range
FROM aircrafts_data
ORDER BY range ASC
LIMIT 1;


In [18]:
%%time
query_df(sql_task_6_a)

CPU times: total: 46.9 ms
Wall time: 65.2 ms


,model,range
0,"{'en': 'Cessna 208 Caravan', 'ru': '??????????...",1200


In [19]:
%%time
query_df(sql_task_6_b)

CPU times: total: 15.6 ms
Wall time: 64.8 ms


,model,range
0,"{'en': 'Cessna 208 Caravan', 'ru': '??????????...",1200


**Комментарий.** Чаще всего второй запрос (`ORDER BY range LIMIT 1`) читается проще. Какой окажется быстрее, зависит от плана выполнения и наличия индекса. Без индекса оба запроса обычно требуют просмотра таблицы. При наличии индекса по `range` поиск может выполниться быстрее.

## Задание 7. Сколько рейсов имеют максимальную длительность полёта? Какова максимальная длительность?

In [21]:
sql_task_7 = """
WITH max_duration AS (
    SELECT MAX(scheduled_arrival - scheduled_departure) AS max_duration
    FROM flights
)
SELECT
    md.max_duration,
    COUNT(*) AS flights_with_max_duration
FROM flights f
CROSS JOIN max_duration md 
WHERE (f.scheduled_arrival - f.scheduled_departure) = md.max_duration
GROUP BY md.max_duration;
"""
print_sql('SQL для задания 7', sql_task_7)
query_df(sql_task_7)

### SQL для задания 7

WITH max_duration AS (
    SELECT MAX(scheduled_arrival - scheduled_departure) AS max_duration
    FROM flights
)
SELECT
    md.max_duration,
    COUNT(*) AS flights_with_max_duration
FROM flights f
CROSS JOIN max_duration md 
WHERE (f.scheduled_arrival - f.scheduled_departure) = md.max_duration
GROUP BY md.max_duration;


,max_duration,flights_with_max_duration
0,0 days 08:50:00,174


CROSS JOIN Каждая строка из flights соединяется с каждой строкой из max_duration

## Задание 8. Уникальные маршруты с максимальной длительностью полёта

In [27]:
sql_task_8 = """
WITH route_durations AS (
    SELECT
        departure_airport,
        arrival_airport,
        MAX(scheduled_arrival - scheduled_departure) AS duration
    FROM flights
    GROUP BY departure_airport, arrival_airport
),
max_route_duration AS (
    SELECT MAX(duration) AS max_duration
    FROM route_durations
)
SELECT DISTINCT
    rd.duration,
    ad.airport_name AS departure_airport_name,
    ad.city AS departure_city,
    aa.airport_name AS arrival_airport_name,
    aa.city AS arrival_city
FROM route_durations rd
JOIN airports_data ad ON ad.airport_code = rd.departure_airport
JOIN airports_data aa ON aa.airport_code = rd.arrival_airport
JOIN max_route_duration mrd ON rd.duration = mrd.max_duration
ORDER BY departure_city, arrival_city;
"""
print_sql('SQL для задания 8', sql_task_8)
query_df(sql_task_8)

### SQL для задания 8

WITH route_durations AS (
    SELECT
        departure_airport,
        arrival_airport,
        MAX(scheduled_arrival - scheduled_departure) AS duration
    FROM flights
    GROUP BY departure_airport, arrival_airport
),
max_route_duration AS (
    SELECT MAX(duration) AS max_duration
    FROM route_durations
)
SELECT DISTINCT
    rd.duration,
    ad.airport_name AS departure_airport_name,
    ad.city AS departure_city,
    aa.airport_name AS arrival_airport_name,
    aa.city AS arrival_city
FROM route_durations rd
JOIN airports_data ad ON ad.airport_code = rd.departure_airport
JOIN airports_data aa ON aa.airport_code = rd.arrival_airport
JOIN max_route_duration mrd ON rd.duration = mrd.max_duration
ORDER BY departure_city, arrival_city;


,duration,departure_airport_name,departure_city,arrival_airport_name,arrival_city
0,0 days 08:50:00,"{'en': 'Domodedovo International Airport', 'ru...","{'en': 'Moscow', 'ru': '????????????'}","{'en': 'Yelizovo Airport', 'ru': '????????????...","{'en': 'Petropavlovsk', 'ru': '???????????????..."
1,0 days 08:50:00,"{'en': 'Domodedovo International Airport', 'ru...","{'en': 'Moscow', 'ru': '????????????'}","{'en': 'Yuzhno-Sakhalinsk Airport', 'ru': '???...","{'en': 'Yuzhno-Sakhalinsk', 'ru': '????????-??..."
2,0 days 08:50:00,"{'en': 'Yelizovo Airport', 'ru': '????????????...","{'en': 'Petropavlovsk', 'ru': '???????????????...","{'en': 'Domodedovo International Airport', 'ru...","{'en': 'Moscow', 'ru': '????????????'}"
3,0 days 08:50:00,"{'en': 'Yuzhno-Sakhalinsk Airport', 'ru': '???...","{'en': 'Yuzhno-Sakhalinsk', 'ru': '????????-??...","{'en': 'Domodedovo International Airport', 'ru...","{'en': 'Moscow', 'ru': '????????????'}"


JOIN с airports_data (ad) — получает название и город аэропорта вылета

JOIN с airports_data (aa) — получает название и город аэропорта прилета

JOIN с max_route_duration (mrd) — оставляет только маршруты, чья длительность равна максимальной

SELECT DISTINCT — убирает возможные дубликаты

ORDER BY — сортирует по городу вылета и прилета


найти колво рейсов у которых продолжительность  полета меньше средней продолжительности полета по всем рейсам



In [63]:
sql_a = """
SELECT COUNT(*) AS flights_below_avg
FROM flights
WHERE (scheduled_arrival - scheduled_departure) < (
    SELECT AVG(scheduled_arrival - scheduled_departure)
    FROM flights
);
"""
print(sql_a)
query_df(sql_a)


SELECT COUNT(*) AS flights_below_avg
FROM flights
WHERE (scheduled_arrival - scheduled_departure) < (
    SELECT AVG(scheduled_arrival - scheduled_departure)
    FROM flights
);



,flights_below_avg
0,19017


Вывести все рейсы (flight_id, flight_no), у которых статус 'Cancelled' (отменен). Ограничить вывод 10 записями.

In [117]:
sql_i = """ 
SELECT * FROM flights
""" 
print(sql_i)
query_df(sql_i)

 
SELECT * FROM flights



,flight_id,flight_no,scheduled_departure,scheduled_arrival,departure_airport,arrival_airport,status,aircraft_code,actual_departure,actual_arrival
0,1185,PG0134,2017-09-10 06:50:00+00:00,2017-09-10 11:55:00+00:00,DME,BTK,Scheduled,319,NaT,NaT
1,3979,PG0052,2017-08-25 11:50:00+00:00,2017-08-25 14:35:00+00:00,VKO,HMA,Scheduled,CR2,NaT,NaT
2,4739,PG0561,2017-09-05 09:30:00+00:00,2017-09-05 11:15:00+00:00,VKO,AER,Scheduled,763,NaT,NaT
3,5502,PG0529,2017-09-12 06:50:00+00:00,2017-09-12 08:20:00+00:00,SVO,UFA,Scheduled,763,NaT,NaT
4,6938,PG0461,2017-09-04 09:25:00+00:00,2017-09-04 10:20:00+00:00,SVO,ULV,Scheduled,SU9,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...
33116,33117,PG0063,2017-08-02 16:25:00+00:00,2017-08-02 17:10:00+00:00,SKX,SVO,Arrived,CR2,2017-08-02 16:25:00+00:00,2017-08-02 17:10:00+00:00
33117,33118,PG0063,2017-07-28 16:25:00+00:00,2017-07-28 17:10:00+00:00,SKX,SVO,Arrived,CR2,2017-07-28 16:30:00+00:00,2017-07-28 17:15:00+00:00
33118,33119,PG0063,2017-09-08 16:25:00+00:00,2017-09-08 17:10:00+00:00,SKX,SVO,Scheduled,CR2,NaT,NaT
33119,33120,PG0063,2017-08-01 16:25:00+00:00,2017-08-01 17:10:00+00:00,SKX,SVO,Arrived,CR2,2017-08-01 16:26:00+00:00,2017-08-01 17:12:00+00:00


In [125]:
sql_i = """ 
SELECT flight_id,	flight_no	 FROM flights 
WHERE status = 'Cancelled'
LIMIT 10;
""" 
print(sql_i)
query_df(sql_i)

 
SELECT flight_id,	flight_no	 FROM flights 
WHERE status = 'Cancelled'
LIMIT 10;



,flight_id,flight_no
0,17173,PG0059
1,652,PG0416
2,714,PG0054
3,807,PG0045
4,859,PG0341
5,935,PG0368
6,1002,PG0335
7,1108,PG0136
8,1192,PG0134
9,1241,PG0005


Найти топ-5 аэропортов по количеству вылетов. Вывести код аэропорта departure_airport, название (на английском) и количество вылетов actual_departure.

In [200]:
sql_svo = """ 
SELECT 
    f.departure_airport AS airport_code,
    a.airport_name->>'en' AS airport_name,
    COUNT(*) AS departures_count
FROM flights f
JOIN airports_data a ON a.airport_code = f.departure_airport
GROUP BY f.departure_airport, a.airport_name
ORDER BY departures_count DESC
LIMIT 5;

""" 
print(sql_svo)
query_df(sql_svo)

 
SELECT 
    f.departure_airport AS airport_code,
    a.airport_name->>'en' AS airport_name,
    COUNT(*) AS departures_count
FROM flights f
JOIN airports_data a ON a.airport_code = f.departure_airport
GROUP BY f.departure_airport, a.airport_name
ORDER BY departures_count DESC
LIMIT 5;




,airport_code,airport_name,departures_count
0,DME,Domodedovo International Airport,3217
1,SVO,Sheremetyevo International Airport,2981
2,LED,Pulkovo Airport,1900
3,VKO,Vnukovo International Airport,1719
4,OVB,Tolmachevo Airport,1055


Для каждого типа самолета (aircraft_code) вывести общее количество проданных билетов COUNT(*)ticket_no и общую выручку SUM(amount). Отсортировать по выручке по убыванию.
Нужно смотреть aircraft_code в flights, ticket_no и amount в ticket_flights

In [220]:
sql_p = """ 
SELECT 
    f.aircraft_code, COUNT(*) AS tickets_sold,
    SUM(tf.amount)  AS revenue
FROM flights f
JOIN ticket_flights tf ON f.flight_id = tf.flight_id
GROUP BY f.aircraft_code
ORDER BY revenue DESC;  
""" 
print(sql_p)
query_df(sql_p)

 
SELECT 
    f.aircraft_code, COUNT(*) AS tickets_sold,
    SUM(tf.amount)  AS revenue
FROM flights f
JOIN ticket_flights tf ON f.flight_id = tf.flight_id
GROUP BY f.aircraft_code
ORDER BY revenue DESC;  



,aircraft_code,tickets_sold,revenue
0,SU9,365698,5114484700.00
1,763,124774,4371277100.00
2,773,144376,3431205500.00
3,319,52853,2706163100.00
4,CR2,150122,1982760500.00
5,321,107129,1638164100.00
6,733,86102,1426552100.00
7,CN1,14672,96373800.00


Найти количество рейсов со статусом 'Delayed'.

In [41]:
sql_b = """
SELECT DISTINCT status FROM flights
""" 
print(sql_b)
query_df(sql_b)


SELECT DISTINCT status FROM flights



,status
0,Departed
1,Arrived
2,On Time
3,Cancelled
4,Delayed
5,Scheduled


количество рейсов для каждого статуса

In [136]:
sql_c = """
SELECT * FROM ticket_flights;
""" 
print(sql_c)
query_df(sql_c)


SELECT * FROM ticket_flights;



,ticket_no,flight_id,fare_conditions,amount
0,0005432159776,30625,Business,42100.00
1,0005435212351,30625,Business,42100.00
2,0005435212386,30625,Business,42100.00
3,0005435212381,30625,Business,42100.00
4,0005432211370,30625,Business,42100.00
...,...,...,...,...
1045721,0005435097522,32094,Economy,5200.00
1045722,0005435097521,32094,Economy,5200.00
1045723,0005435104384,32094,Economy,5200.00
1045724,0005435104352,32094,Economy,5200.00


In [59]:
sql_c =  """
SELECT status, COUNT(*) AS flights_count
FROM flights
GROUP BY status
ORDER BY flights_count;
""" 
print(sql_c)
query_df(sql_c)


SELECT status, COUNT(*) AS flights_count
FROM flights
GROUP BY status
ORDER BY flights_count;



,status,flights_count
0,Delayed,41
1,Departed,58
2,Cancelled,414
3,On Time,518
4,Scheduled,15383
5,Arrived,16707


In [54]:
sql_b = """
SELECT  COUNT (*) AS flights_count
FROM flights
WHERE status = 'Cancelled';
""" 
print(sql_b)
query_df(sql_b)


SELECT  COUNT (*) AS flights_count
FROM flights
WHERE status = 'Cancelled';



,flights_count
0,414


## Задание 9. На каком аэропорту лежит максимальная нагрузка по отправлениям и прибытиям?

In [21]:
sql_task_9 = """
WITH airport_load AS (
    SELECT departure_airport AS airport_code, COUNT(*) AS cnt
    FROM flights
    GROUP BY departure_airport
    UNION ALL
    SELECT arrival_airport AS airport_code, COUNT(*) AS cnt
    FROM flights
    GROUP BY arrival_airport
),
agg AS (
    SELECT airport_code, SUM(cnt) AS total_load
    FROM airport_load
    GROUP BY airport_code
)
SELECT
    a.airport_name,
    a.city,
    agg.total_load
FROM agg
JOIN airports_data a ON a.airport_code = agg.airport_code
ORDER BY agg.total_load DESC
LIMIT 1;
"""
print_sql('SQL для задания 9', sql_task_9)
query_df(sql_task_9)

### SQL для задания 9

WITH airport_load AS (
    SELECT departure_airport AS airport_code, COUNT(*) AS cnt
    FROM flights
    GROUP BY departure_airport
    UNION ALL
    SELECT arrival_airport AS airport_code, COUNT(*) AS cnt
    FROM flights
    GROUP BY arrival_airport
),
agg AS (
    SELECT airport_code, SUM(cnt) AS total_load
    FROM airport_load
    GROUP BY airport_code
)
SELECT
    a.airport_name,
    a.city,
    agg.total_load
FROM agg
JOIN airports_data a ON a.airport_code = agg.airport_code
ORDER BY agg.total_load DESC
LIMIT 1;


,airport_name,city,total_load
0,"{'en': 'Domodedovo International Airport', 'ru...","{'en': 'Moscow', 'ru': '????????????'}",6434


UNION ALL — объединяет два набора данных (вылеты и прилеты)

AS = alias()

## Задание 10. Среднее количество мест в самолётах по каждому классу обслуживания

In [2]:
sql_d = """ 
SELECT 
FROM seats;
"""
query_df(sql_d)

NameError: name 'query_df' is not defined

найти самолет с самым маленьким колвом мест

In [13]:
sql_j = """
WITH seats_per_aircraft AS (
    SELECT 
        aircraft_code,
        COUNT(*) AS seat_count
    FROM seats
    GROUP BY aircraft_code
)
SELECT 
    aircraft_code,
    seat_count
FROM seats_per_aircraft
ORDER BY seat_count ASC
LIMIT 1;
"""  
query_df(sql_j)

,aircraft_code,seat_count
0,CN1,12


In [20]:
sql_task_10 = """
WITH seats_per_aircraft AS (
    SELECT
        aircraft_code,
        COUNT(*) AS seat
    FROM seats
    GROUP BY aircraft_code
)
SELECT
    aircraft_code,
    seat
FROM seats_per_aircraft
ORDER BY seat
LIMIT 1;
"""
print_sql('SQL для задания 10', sql_task_10)
query_df(sql_task_10)

### SQL для задания 10

WITH seats_per_aircraft AS (
    SELECT
        aircraft_code,
        COUNT(*) AS seat
    FROM seats
    GROUP BY aircraft_code
)
SELECT
    aircraft_code,
    seat
FROM seats_per_aircraft
ORDER BY seat
LIMIT 1;


,aircraft_code,seat
0,CN1,12


AVG(seat_count) — может вернуть numeric или integer в зависимости от СУБД

numeric — приводит к числовому типу (явное преобразование)

ROUND(2) — округляет до 2 десятичных знаков

## Задание 11. Самый дорогой перелёт

In [23]:
sql_task_11 = """
WITH flight_revenue AS (
    SELECT
        tf.flight_id,
        SUM(tf.amount) AS final_amount
    FROM ticket_flights tf
    GROUP BY tf.flight_id
)
SELECT
    fr.flight_id,
    fr.final_amount,
    dep.airport_name AS departure_airport,
    dep.city AS departure_city,
    arr.airport_name AS arrival_airport,
    arr.city AS arrival_city
FROM flight_revenue fr
JOIN flights f ON f.flight_id = fr.flight_id
JOIN airports_data dep ON dep.airport_code = f.departure_airport
JOIN airports_data arr ON arr.airport_code = f.arrival_airport
ORDER BY fr.final_amount DESC
LIMIT 1;
"""
print_sql('SQL для задания 11', sql_task_11)
query_df(sql_task_11)

### SQL для задания 11

WITH flight_revenue AS (
    SELECT
        tf.flight_id,
        SUM(tf.amount) AS final_amount
    FROM ticket_flights tf
    GROUP BY tf.flight_id
)
SELECT
    fr.flight_id,
    fr.final_amount,
    dep.airport_name AS departure_airport,
    dep.city AS departure_city,
    arr.airport_name AS arrival_airport,
    arr.city AS arrival_city
FROM flight_revenue fr
JOIN flights f ON f.flight_id = fr.flight_id
JOIN airports_data dep ON dep.airport_code = f.departure_airport
JOIN airports_data arr ON arr.airport_code = f.arrival_airport
ORDER BY fr.final_amount DESC
LIMIT 1;


,flight_id,final_amount,departure_airport,departure_city,arrival_airport,arrival_city
0,2354,17146600.00,"{'en': 'Domodedovo International Airport', 'ru...","{'en': 'Moscow', 'ru': '????????????'}","{'en': 'Khabarovsk-Novy Airport', 'ru': '?????...","{'en': 'Khabarovsk', 'ru': '??????????????????'}"


In [24]:
sql_task_11_explain = 'EXPLAIN ANALYZE ' + sql_task_11
query_df(sql_task_11_explain)

,QUERY PLAN
0,Limit (cost=21096.55..21096.55 rows=1 width=2...
1,-> Sort (cost=21096.55..21134.24 rows=1507...
2,Sort Key: (sum(tf.amount)) DESC
3,Sort Method: top-N heapsort Memory: 26kB
4,-> Hash Join (cost=20128.65..21021.1...
5,Hash Cond: (f.arrival_airport = ...
6,-> Hash Join (cost=20123.31..2...
7,Hash Cond: (f.departure_ai...
8,-> Hash Join (cost=20117...
9,Hash Cond: (f.flight...


**Краткий анализ EXPLAIN ANALYZE:**
- если увидишь `Seq Scan`, значит PostgreSQL читает всю таблицу целиком;
- для ускорения запроса полезны индексы по `ticket_flights.flight_id`, `flights.flight_id`, `flights.departure_airport`, `flights.arrival_airport`;
- основной дорогой участок обычно связан с агрегацией `SUM(amount)` по `ticket_flights`.

## Дополнительное задание. 3 запроса для поиска узких мест авиаперевозчика

## Топ-10 самых задерживающихся маршрутов ##

In [25]:
sql_extra_1 = """
SELECT
    departure_airport,
    arrival_airport,
    COUNT(*) AS delayed_flights
FROM flights
WHERE status = 'Delayed'
GROUP BY departure_airport, arrival_airport
ORDER BY delayed_flights DESC
LIMIT 10;
"""
query_df(sql_extra_1)

,departure_airport,arrival_airport,delayed_flights
0,AER,IWA,1
1,ARH,PEE,1
2,BAX,ASF,1
3,BZK,DME,1
4,DME,BZK,1
5,DME,CEK,1
6,DME,KVX,1
7,DME,LED,1
8,DME,PKV,1
9,ABA,OVB,1


## Топ 10 самых прибыльных рейсов ##

In [26]:
sql_extra_2 = """
SELECT
    f.flight_no,
    f.departure_airport,
    f.arrival_airport,
    SUM(tf.amount) AS revenue
FROM flights f
JOIN ticket_flights tf ON tf.flight_id = f.flight_id
GROUP BY f.flight_no, f.departure_airport, f.arrival_airport
ORDER BY revenue DESC
LIMIT 10;
"""
query_df(sql_extra_2)

,flight_no,departure_airport,arrival_airport,revenue
0,PG0208,DME,KHV,753478300.00
1,PG0209,KHV,DME,733797800.00
2,PG0222,DME,OVB,548218900.00
3,PG0223,OVB,DME,531503700.00
4,PG0357,KHV,LED,507672400.00
5,PG0356,LED,KHV,498750700.00
6,PG0277,SVO,OVB,458130400.00
7,PG0278,OVB,SVO,446051200.00
8,PG0198,LED,IKT,411603000.00
9,PG0199,IKT,LED,404050300.00


In [27]:
sql_extra_3 = """
WITH sold_seats AS (
    SELECT flight_id, COUNT(*) AS sold_cnt
    FROM boarding_passes
    GROUP BY flight_id
)
SELECT
    f.aircraft_code,
    ROUND(AVG(ss.sold_cnt)::numeric, 2) AS avg_sold_seats
FROM flights f
JOIN sold_seats ss ON ss.flight_id = f.flight_id
GROUP BY f.aircraft_code
ORDER BY avg_sold_seats DESC;
"""
query_df(sql_extra_3)

,aircraft_code,avg_sold_seats
0,773,264.93
1,763,113.94
2,321,88.81
3,733,80.26
4,SU9,56.81
5,319,53.58
6,CR2,21.48
7,CN1,6.00


Найти топ5 убыточных рейсов, вывести откуда куда летел самолет(аэропорт), сколько было свободных мест?
Узнать количество проданных билетов 

In [34]:
sql_o = """ 
SELECT 
    f.flight_id,
    f.departure_airport,
    f.arrival_airport,
    COUNT(DISTINCT bp.boarding_no) AS sold_tickets,
    s.total_seats,
    s.total_seats - COUNT(DISTINCT bp.boarding_no) AS free_seats
FROM flights f
JOIN (
    SELECT aircraft_code, COUNT(*) AS total_seats
    FROM seats
    GROUP BY aircraft_code
) s ON s.aircraft_code = f.aircraft_code
LEFT JOIN boarding_passes bp ON bp.flight_id = f.flight_id
GROUP BY f.flight_id, f.departure_airport, f.arrival_airport, s.total_seats
ORDER BY free_seats DESC
LIMIT 5;
""" 
print(sql_o)
query_df(sql_o)

 
SELECT 
    f.flight_id,
    f.departure_airport,
    f.arrival_airport,
    COUNT(DISTINCT bp.boarding_no) AS sold_tickets,
    s.total_seats,
    s.total_seats - COUNT(DISTINCT bp.boarding_no) AS free_seats
FROM flights f
JOIN (
    SELECT aircraft_code, COUNT(*) AS total_seats
    FROM seats
    GROUP BY aircraft_code
) s ON s.aircraft_code = f.aircraft_code
LEFT JOIN boarding_passes bp ON bp.flight_id = f.flight_id
GROUP BY f.flight_id, f.departure_airport, f.arrival_airport, s.total_seats
ORDER BY free_seats DESC
LIMIT 5;



,flight_id,departure_airport,arrival_airport,sold_tickets,total_seats,free_seats
0,252,DME,OVB,0,402,402
1,253,DME,OVB,0,402,402
2,248,DME,OVB,0,402,402
3,246,DME,OVB,0,402,402
4,254,DME,OVB,0,402,402


In [30]:
sql_m = """
SELECT 
    flight_id, COUNT(*) ticket_no

FROM ticket_flights;
""" 
print(sql_m)
query_df(sql_m)


SELECT 
    flight_id, COUNT(*) ticket_no

FROM ticket_flights;



GroupingError: column "ticket_flights.flight_id" must appear in the GROUP BY clause or be used in an aggregate function
LINE 3:     flight_id, COUNT(*) ticket_no
            ^


## Итог

Этот ноутбук можно запускать по ячейкам сверху вниз. Если нужно сдавать именно `.ipynb`, используй этот файл. Если удобнее запускать как обычный Python-скрипт, рядом подготовлен файл `lab_solution.py`.